# Answer Strategy (MIDA III): Estimators and Regression

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/honr_464_logo.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>HONR 46400 — Evidence-Driven Research</center>
# <center>Professor: Davi Moreira</center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/blob/main/notebooks/student/nb09_answer_strategy_student.ipynb)

---

## 🧭 Inquiry & Claim Boundary

**Inquiry emphasis:** the **A** of MIDA (the Answer strategy) — the cross-cutting
machinery every compass position leans on, not a position of its own. Your Model
said what the world could be; your Inquiry named the exact quantity you want and
how far it must reach; your Data strategy decided who you get to see. The answer
strategy is the last letter: the **procedure** that turns data into a number, and
the honest accounting of how far that number can be trusted — the machinery that
later licenses generalization, prediction, and causal reasoning alike.

| | |
|---|---|
| **A claim this topic PERMITS** | "My answer strategy is [procedure]; applied to my data it yields [estimate ± uncertainty] of my declared inquiry." Or, descriptively: "adjusting for the variables I named, the association between X and Y is [estimate ± uncertainty]." |
| **A claim this topic does NOT permit** | Reading a regression coefficient as a causal effect because covariates were "controlled for" — controlling for is not identifying (the descriptive→causal crossing, unpaid). |

**Where this sits in the course:** meetings 18–19 of 44. It closes MIDA (Model,
Inquiry, Data, **Answer**) and builds milestone M06 (your Answer Strategy
Declaration). It uses the same mentoring-program world you first met in nb04, and
it is the machinery every later compass position — generalization, prediction,
causal reasoning — will lean on.

*Provenance: RDSS ch. 9 'Choosing an answer strategy' (incl. §9.1.3 'Procedure') + declaration_9.1 | the estimand / estimator / estimate triad, regression as an answer strategy, and the answer strategy as the whole if-then procedure | difference-in-means and estimatr::lm_robust translated to hand-rolled pandas and statsmodels OLS with HC2; the multiple-bites simulation built on §9.1.3 | translated (R→Python) + newly-constructed-from-source-concept*

## Learning Objectives

By the end of this notebook, you will be able to:

1. Name and distinguish the **estimand / estimator / estimate** triad — the
   genie's number, the recipe, and the dish you actually cooked.
2. Hand-roll a **difference-in-means** estimator (with its standard error) on a
   simulated experiment, then unhide the truth and **measure the miss**.
3. Rebuild that same estimator as an **OLS regression**, and add **covariate
   adjustment** to buy precision when assignment is random.
4. Fit an adjusted association on real survey data with **robust (HC2) standard
   errors**, and read the coefficient strictly **descriptively**.
5. Explain why **"controlling for" is not "identifying,"** and refuse the causal
   overread of an adjusted coefficient.
6. Sketch the **answer strategy for your own project** (outcome, key regressor,
   two honest covariates) into the M06 template.

---

In [ ]:
# Setup — run this cell first.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)
plt.rcParams["figure.figsize"] = (9, 5)

SEED = 464  # course number — keeps every simulation reproducible
rng = np.random.default_rng(SEED)

# Data loads: GitHub raw URL first (works in Colab), local repo path as fallback.
DATA_URL = ("https://raw.githubusercontent.com/davi-moreira/"
            "2026F_evidence_driven_research_purdue_HONR464/main/notebooks/data/")

def load_course_data(filename):
    """Load a course dataset from GitHub, falling back to the local repo copy."""
    try:
        return pd.read_csv(DATA_URL + filename)
    except Exception:
        from pathlib import Path
        local = Path("notebooks/data") / filename
        if not local.exists():
            local = Path("../data") / filename
        return pd.read_csv(local)

print("✓ Setup complete — seed", SEED)

## 1. Why This Matters — The Recipe, the Dish, and the Genie's Number

> *"When you tell me your result, I have three questions, always in this order:
> what number were you actually trying to estimate, what exact procedure produced
> the number you are showing me, and how far off could that number be? If you can
> answer all three, I trust the fourth thing — the number itself. If you cannot, I
> do not care what it says."*
> — a **thesis advisor** reading your first results section

Friday's async module ended on three words that name those first questions. Here
they are again, sharpened, with the cooking metaphor that will carry all week:

- **Estimand** — the true quantity you are after, the number the world actually
  holds. *The dish the genie would serve you.* You almost never get to see it.
- **Estimator** — the *recipe*: the exact, repeatable procedure you apply to data
  to produce an answer.
- **Estimate** — the *dish you actually cooked*: the one specific number your
  recipe returns on your data.

The whole discipline of an answer strategy is keeping these three straight. The
Beacon's breakfast story failed exactly here: it served an **estimate** (a
difference in failure rates) as though it were the **estimand** (a causal effect).
Today you get something a real study never gets — a world where you plant the
estimand yourself. That lets you run your recipe, then lift the lid on the genie's
dish and measure precisely how far your estimate landed from the truth.

> **A question that often comes up here:** *"If real research never sees the
> estimand, what is the point of a simulated world where we do?"* It is a flight
> simulator. You cannot learn whether a recipe is trustworthy by running it once on
> real data where the truth is hidden — you would have nothing to compare against.
> So you build a world whose truth you set, run the recipe there, and watch how it
> behaves. Only a recipe that recovers a truth you *can* check earns your trust on
> the day the truth is hidden.

---

## 2. The Mentoring-Program World from nb04 Returns

**Guiding question:** *When you can secretly see the true effect, how close does an honest recipe actually land — and what does the gap teach you?*

Recall the world you built in nb04: a mentoring program, and for each person a
**belonging score** on a 0–100 scale in two possible versions — `Y0`, their score
**without** a mentor, and `Y1`, their score **with** one. The program adds a
constant **effect** of 2.0 points to everyone. Because we built the world, we can
write both columns for every person — the "science-fiction" view no real study
ever has.

The cell below re-creates that world for 500 people and computes the **estimand**:
the average treatment effect, the mean of `Y1 − Y0` across everyone. This is the
genie's number — the dish we are trying to recover.

> 💡 **Gemini Prompt:** "Here is a Python cell that builds a simulated 'both
> potential outcomes' world — each person has a Y0 (belonging score without a
> mentor) and a Y1 (with one) — and computes the true average treatment effect:
> [paste the next cell]. Explain what a 'potential outcome' is and why a real study
> can never observe both columns for the same person."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Confirm the estimand your cell printed is exactly 2.0, and check that matches
>   Gemini's account of how Y1 is built from Y0 plus the constant effect.
> - [ ] Make sure Gemini calls this a *simulated* truth you planted, not something
>   estimated from data — the whole point is that you set it yourself.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# The mentoring-program world from nb04 — both potential outcomes visible.
def make_world(n=100, effect=2.0, noise=2.0, rng=rng):
    """The mentoring-program world: each person's belonging score (0-100)
    without a mentor (Y0) and with one (Y1 = Y0 + effect)."""
    Y0 = rng.normal(50, 10, n) + rng.normal(0, noise, n)
    return pd.DataFrame({"Y0": Y0.round(1), "Y1": (Y0 + effect).round(1)})

world = make_world(n=500)                       # 500 people, both columns known
TRUE_ATE = (world["Y1"] - world["Y0"]).mean()   # the ESTIMAND — the genie's number

print("The world (first 5 people) — the science-fiction view:")
print(world.head().to_string())
print()
print(f"ESTIMAND (true average treatment effect) = {TRUE_ATE:.4f}")

# Self-check: the effect is constant at 2.0 by construction, so the ATE must be 2.0.
assert abs(TRUE_ATE - 2.0) < 0.01, "the estimand should be 2.0 by construction"
print("✓ Self-check passed: the genie's number is 2.0 — hold onto it, we will hide it.")

Now leave science fiction and enter reality. In any real study you observe **one**
outcome per person: either their with-mentor score or their without-mentor score,
never both. So we run an **experiment**: randomly assign half the 500 to the
mentoring program, then reveal only the outcome each person's assignment lets us
see. From that half-hidden data, our recipe must try to recover the 2.0 we just
computed — without being allowed to look at it.

### 🔮 Pause & Predict

We are about to randomly assign 250 of the 500 people to mentoring, reveal each
person's single observed score, and apply the **difference-in-means** recipe:
average belonging among the mentored minus average belonging among the rest. The
true effect is exactly 2.0. **Before running the next cell**, commit a prediction:
how close to 2.0 will the recipe land — will it hit 2.0 on the nose, miss by a
little, or miss by a lot? Write one sentence on why.

### YOUR ANSWER HERE:

**My prediction — how close the recipe lands to 2.0, and why:**

---

### 🛠️ Run the Study: assign, reveal, and estimate

Run the two cells below. The first performs **complete random assignment** (exactly
250 treated, by a seeded shuffle) and reveals the observed outcome `Y`. The second
defines the difference-in-means recipe — its point estimate *and* its standard
error, the honest measure of how much the estimate would wobble across repeated
experiments — and applies it.

> 💡 **Gemini Prompt:** "These two cells randomly assign 250 of 500 people to
> treatment, reveal only each person's observed outcome, then estimate the effect as
> the treated mean minus the control mean, with a standard error: [paste both
> cells]. Explain in plain language what the standard error is measuring and why a
> single experiment's estimate is not expected to hit the true value exactly."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check that the estimate and standard error Gemini discusses are the actual
>   numbers your cell printed (about 2.04 and 0.9), not stand-in values.
> - [ ] Confirm from the output that the 95% interval really does cover the true 2.0
>   — and treat Gemini's interpretation as a prompt to check, not a fact to copy.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# Complete random assignment: exactly 250 of 500 treated, by a seeded shuffle.
n = len(world)
z = np.zeros(n, dtype=int)
z[: n // 2] = 1
world = world.assign(Z=rng.permutation(z))

# Reveal ONLY the outcome each assignment lets us see (the real-world view).
world["Y"] = np.where(world["Z"] == 1, world["Y1"], world["Y0"])

print("What a real study actually observes (Y0/Y1 now hidden from the recipe):")
print(world[["Z", "Y"]].head().to_string())
print()
print(f"  treated (Z=1): {int(world['Z'].sum())} people")
print(f"  control (Z=0): {int((world['Z'] == 0).sum())} people")

In [ ]:
# The difference-in-means estimator — the recipe (RDSS ch. 9 / API_MAPPING pattern).
def difference_in_means(df, y="Y", z="Z"):
    """Estimate = mean(Y | treated) - mean(Y | control); SE from the two groups."""
    g1, g0 = df.loc[df[z] == 1, y], df.loc[df[z] == 0, y]
    est = g1.mean() - g0.mean()
    se = np.sqrt(g1.var(ddof=1) / len(g1) + g0.var(ddof=1) / len(g0))
    return est, se

est, se = difference_in_means(world)
ci_low, ci_high = est - 1.96 * se, est + 1.96 * se

print("ESTIMATE from the difference-in-means recipe:")
print(f"  estimate = {est:.4f}   (standard error {se:.4f})")
print(f"  95% interval: [{ci_low:.3f}, {ci_high:.3f}]")
print()
# Now lift the lid on the genie's dish and measure the miss.
print(f"  ESTIMAND (the hidden truth) = {TRUE_ATE:.4f}")
print(f"  the recipe MISSED by {est - TRUE_ATE:+.4f}")

assert round(est, 4) == 2.0372, "estimate drifted — check the seed / cell order"
assert ci_low <= TRUE_ATE <= ci_high, "the interval should cover the truth"
print("✓ Self-check passed: the estimate (2.04) missed the truth (2.0) by only 0.04,")
print("  and its 95% interval comfortably covers 2.0.")

### 🔍 Reading the Evidence

The recipe returned 2.04 when the truth was 2.0 — a miss of about four hundredths.
In the cell below, write two things. First: was your prediction closer to "hits 2.0
exactly" or "misses by a lot," and what does a miss of only 0.04 (with a standard
error near 0.9) tell you about whether landing *exactly* on 2.0 was ever the right
expectation? Second: the interval [0.34, 3.73] is wide — it covers the truth but
also covers 3.0 and 1.0. Name the one design change from earlier units that would
tighten it, and say why a single estimate should never be reported as if it were
the estimand.

### YOUR ANSWER HERE:

**My prediction vs the 0.04 miss — what "exactly 2.0" got wrong:**

**Why the wide interval matters, and what would tighten it:**

---

The three words now have a home. On this experiment: the **estimand** is 2.0 (the
genie's number, which we normally never see); the **estimator** is the
difference-in-means recipe (a procedure you could hand to anyone); and the
**estimate** is 2.04 (the one dish this data cooked). Matching the recipe to the
inquiry is the skill — difference-in-means answers "what is the average effect?"
because random assignment makes the two groups comparable. A recipe aimed at a
different inquiry would return a different, and possibly meaningless, dish.

> **A question that often comes up here:** *"Two people run two estimators on the
> same data and get two different numbers. Who is right?"* Possibly both — if they
> declared **different inquiries**. An estimate is only right or wrong relative to
> the estimand it targets. The danger is not disagreement; it is an estimator
> quietly answering a question no one declared, then having its dish served as the
> answer to the question everyone assumed. Declare the inquiry first (you did, in
> nb04), and the estimator has something to be faithful to.

### 📝 Practice

Match each inquiry to the answer strategy that fits it, and **flag the one
mismatch** — the pairing where the named strategy cannot deliver the inquiry.
Answer in the cell that follows.

- **A.** Inquiry: the average effect of a tutoring program on test scores, in a
  randomized trial. → Strategy: difference-in-means between treated and control.
- **B.** Inquiry: the share of first-years who feel they belong. → Strategy: a
  sample proportion with an interval.
- **C.** Inquiry: the association between study hours and GPA, holding major fixed.
  → Strategy: a regression of GPA on study hours and major.
- **D.** Inquiry: the *causal* effect of a new advising model on retention. →
  Strategy: compare retention between the schools that *chose* to adopt it and
  those that did not (no randomization, no adjustment).


### YOUR ANSWER HERE:

**A:** &nbsp; **B:** &nbsp; **C:** &nbsp; **D:**

**The mismatch, and why the strategy cannot deliver the inquiry:**

---

### 🎟️ Claim Ticket

**Claim Ticket #18** — in one sentence each, name (a) **your project's inquiry**
(the quantity you declared in M03), (b) **your candidate answer strategy** (the
recipe you would apply), and (c) **why they match** — what makes that recipe
faithful to that inquiry.

### YOUR ANSWER HERE:

**(a) My inquiry:**

**(b) My candidate answer strategy:**

**(c) Why they match:**

---

---

*(Meeting M19 picks up here.)*

## 3. Regression as an Estimator — Not an Oracle

**Guiding question:** *Is regression a more powerful, more "causal" tool than a plain difference in means — or the same recipe with room to adjust?*

> *"Every few weeks I read a paper that 'controlled for' a dozen variables and then
> announced an effect. Controlling for things is not a magic word. It is an
> arithmetic operation with a precise, limited meaning — and the moment an author
> forgets that, the coefficient stops being evidence and starts being wishful
> thinking."*
> — a **journal reviewer** who has seen "we controlled for everything" one time too many

**Regression** is the most-used answer strategy in the social sciences, and the
most misunderstood. Today you will see it demystified in two moves. First: on a
randomized experiment, a regression of the outcome on the treatment is *literally
the same recipe* as difference-in-means — same number, exactly. Regression is not a
more powerful oracle; it is difference-in-means with room to add adjustments.

Second: adding a **covariate** — a pre-treatment variable that predicts the
outcome — can *tighten* the estimate when assignment is random, because it soaks up
outcome variance that would otherwise show up as noise. That is a real gift. But it
is a gift of **precision**, not of **identification** — and telling those two apart
is the whole boundary of this notebook.

### 🔮 Pause & Predict

We will fit an ordinary regression of belonging `Y` on the treatment `Z` in the
mentoring experiment, using robust standard errors. You already have the
difference-in-means estimate for this exact data: 2.0372. **Before running**,
predict: will the regression's coefficient on `Z` be (a) exactly 2.0372, (b) close
but a bit different, or (c) quite different? Commit to one, and say why.

### YOUR ANSWER HERE:

**My prediction (a / b / c) and why:**

---

### 🛠️ Hands-On: regression, twice

Run the cells below. The first fits `Y ~ Z` with **HC2 robust standard errors**
(the course keeps this default because it matches the book's `estimatr::lm_robust`
toolchain, so numbers line up) and compares the coefficient to your hand-rolled
difference-in-means. The second adds a **baseline** covariate — a noisy belonging
"pulse" measured on everyone *before* the program — and shows what adjustment buys.

> 💡 **Gemini Prompt:** "The first cell regresses an outcome Y on a treatment Z, and
> the second adds a pre-treatment covariate called baseline: [paste both cells].
> Explain why regressing Y on Z alone returns the exact same number as a difference
> in means, and how adding a pre-treatment covariate can shrink the standard error
> without changing the story."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Confirm from your output that the OLS coefficient on Z really equals your
>   hand-rolled difference-in-means to the decimal — the cell asserts it, so read the
>   printed match yourself.
> - [ ] Check that the standard error actually shrank after adjustment in your
>   printout, and make sure Gemini ties that gain to assignment being *random*, not
>   to adjustment being magic.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
import statsmodels.formula.api as smf

def report(fit, term):
    """Print only the three numbers that matter: coefficient, robust SE, 95% CI."""
    b, se = fit.params[term], fit.bse[term]
    print(f"  {term:>10s}: coef = {b:6.4f}   SE = {se:6.4f}   "
          f"95% CI [{b - 1.96*se:.3f}, {b + 1.96*se:.3f}]")

# Move 1: regression of Y on Z IS difference-in-means.
fit_dim = smf.ols("Y ~ Z", data=world).fit(cov_type="HC2")
print("Regression  Y ~ Z  (HC2 robust SEs):")
report(fit_dim, "Z")
print(f"\n  hand-rolled difference-in-means was: estimate = {est:.4f}, SE = {se:.4f}")
assert round(fit_dim.params["Z"], 4) == round(est, 4), "OLS on Z must equal diff-in-means"
print("✓ Same recipe: the OLS coefficient on Z equals difference-in-means exactly.")

In [ ]:
# Move 2: add a pre-treatment covariate and watch PRECISION improve.
# 'baseline' = a noisy belonging pulse taken BEFORE anyone was assigned. Because it
# is measured pre-program, it is a legitimate covariate (it cannot be an effect of
# treatment); it is a noisy echo of each person's untreated level.
world["baseline"] = (world["Y0"] + rng.normal(0, 5, n)).round(1)

# Random assignment balanced the covariate — check it before trusting the adjustment.
bal = world.groupby("Z")["baseline"].mean().round(2)
print(f"baseline mean — control: {bal[0]}, treated: {bal[1]}  (balanced, as random assignment promises)")
print()

fit_adj = smf.ols("Y ~ Z + baseline", data=world).fit(cov_type="HC2")
print("Unadjusted   Y ~ Z            (HC2):")
report(fit_dim, "Z")
print("Adjusted     Y ~ Z + baseline (HC2):")
report(fit_adj, "Z")

assert fit_adj.bse["Z"] < fit_dim.bse["Z"], "adjustment should shrink the SE here"
print(f"\n✓ Self-check passed: adjustment cut the standard error from "
      f"{fit_dim.bse['Z']:.3f} to {fit_adj.bse['Z']:.3f} — same story, more precisely told.")

The adjusted estimate (about 1.85) and the unadjusted one (2.04) tell the *same
story* — a positive effect near 2 — but the adjusted standard error is roughly half
the size. That shrink is the entire payoff of adjustment here, and it was safe to
take **only because assignment was random**, which balanced the baseline across the
two groups (control 50.03, treated 50.27). The covariate did not rescue a broken
comparison; it sharpened an already-fair one. Keep that clause — *only because
assignment was random* — because in a moment we meet data where it does not hold.

How much precision a covariate buys depends on how much of the outcome it explains:

<center><img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/rdss_fig_18_2.png" width="80%"/></center>

*The better a pre-treatment covariate predicts the outcome (higher R-squared), the
more statistical power (left) and the smaller the true standard error (right) the
same design earns at every sample size — that is the precision adjustment buys. Note
what it does not buy: identification. On observational data a tighter estimate of a
confounded association is still a confounded association. (Figure from the
replication materials of Blair, Coppock & Humphreys (2023),* Research Design in the
Social Sciences *— MIT-licensed archive; the book is free at book.declaredesign.org.)*

> **A question that often comes up here:** *"So adjusting for more variables always
> makes my estimate better?"* No — and this is the trap the reviewer was warning
> about. On a *randomized* experiment, adjusting for pre-treatment covariates buys
> precision. On *observational* data, adjustment does something much more
> dangerous: it changes the number and tempts you to call the result causal. More
> covariates can even make things worse (adjusting for something on the causal path,
> or for a collider). Adjustment is a scalpel, not a cleanse.

## 4. The Same Machinery on Real Data — and the Boundary It Cannot Cross

**Guiding question:** *When nobody was randomly assigned, what is an adjusted regression coefficient actually allowed to mean?*

The mentoring world was randomized, so we knew what the truth was and could trust
the comparison. Real survey data is different: nobody was randomly assigned to
anything. Watch the *identical* regression machinery run on the LAPOP Brazil survey
— and watch how carefully we must word what comes out.

We will model **trust in the police** (1–7) as a function of perceived **government
responsiveness** (1–7), adjusting for **ideology** (1–10). Same recipe as before.
The difference is entirely in what we are allowed to *say* about the coefficient.

> 💡 **Gemini Prompt:** "This cell fits the same kind of regression as before, but
> on observational survey data with no random assignment — trust in police on
> perceived government responsiveness, adjusting for ideology: [paste the next
> cell]. List the reasons an adjusted coefficient from observational data cannot be
> read as a causal effect, even though the arithmetic is identical to the
> experiment's."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Confirm the coefficient Gemini reasons about is the roughly 0.24 your cell
>   actually printed, and that it frames the reading as descriptive, not causal.
> - [ ] For every "unmeasured confounder" it lists, decide yourself whether that
>   thing plausibly moves both responsiveness perceptions and police trust — you own
>   that judgment, not the AI.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# Real observational data: no random assignment anywhere in sight.
lapop = load_course_data("lapop_brazil.csv")
assert lapop.shape == (10000, 10), "unexpected shape — flag this!"
print("✓ Loaded lapop_brazil.csv —", lapop.shape[0], "rows,", lapop.shape[1], "columns")
print()

# First: classical vs robust SEs on the simple model — why the course uses HC2.
simple_classical = smf.ols("trust_police ~ govt_responsive", data=lapop).fit()
simple_robust = smf.ols("trust_police ~ govt_responsive", data=lapop).fit(cov_type="HC2")
print("trust_police ~ govt_responsive — same coefficient, two standard errors:")
print(f"  coefficient       = {simple_robust.params['govt_responsive']:.4f}")
print(f"  classical SE      = {simple_classical.bse['govt_responsive']:.4f}")
print(f"  robust (HC2) SE   = {simple_robust.bse['govt_responsive']:.4f}   ← the course default")
print()

# Now the adjusted descriptive model.
fit_lapop = smf.ols("trust_police ~ govt_responsive + ideology", data=lapop).fit(cov_type="HC2")
print("trust_police ~ govt_responsive + ideology  (HC2 robust SEs):")
report(fit_lapop, "govt_responsive")
report(fit_lapop, "ideology")
print(f"  n = {int(fit_lapop.nobs)},  R-squared = {fit_lapop.rsquared:.3f}")

assert 0.20 < fit_lapop.params["govt_responsive"] < 0.28, "coef drifted — investigate"
print("\n✓ Self-check passed: adjusting for ideology, a one-point-higher government-")
print("  responsiveness rating goes with about 0.24 more points of police trust.")

### A quick primer: what an association looks like

Before you read the coefficient aloud, calibrate your eye. An **association** has a
direction and a strength, and the correlation coefficient r is the standard
shorthand for both — one number from −1 to +1:

<center><img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/correlation_gallery.png" width="85%"/></center>

*Reading r: near ±1 the cloud tightens to a line (strong linear association), near 0
it is a shapeless blob (weak or none), and the sign gives the direction. A regression
coefficient reports the same kind of relationship — an association — but in the
variables' own units rather than on the −1-to-+1 scale, which is why the sentence you
are about to write says "associated," never "caused." (From Professor Moreira's QM
67000 Business Analytics slides.)*

### Read it aloud — descriptively, then strike out the overreach

The coefficient on `govt_responsive` is about **0.24**. Here is the ONLY reading the
design permits — a strictly descriptive, associational one:

> *"Among these 10,000 resampled LAPOP Brazil interviews, comparing respondents who
> differ by one point in perceived government responsiveness but sit at the same
> point on the ideology scale, average reported trust in the police is about 0.24
> points higher on the 1–7 scale."*

Now the sentence it is so tempting to write instead — and why it is **forbidden**
here. Cross it out in your mind as you read it:

> ~~"Making the government one point more responsive causes trust in the police to
> rise by 0.24 points."~~

Nothing in this analysis earns that verb. No one was randomly assigned a level of
government responsiveness; people who perceive more responsiveness differ from
those who perceive less in a hundred unmeasured ways (experiences with officials,
region, income, personality), and any of those could drive both perceptions at
once. Adjusting for ideology closes exactly *one* of those doors and leaves the rest
wide open. That is the boundary lesson in five words: **controlling for is not
identifying.** Adjustment removes the influence of the variables you *named*; it can
do nothing about the ones you did not — and in observational data there are always
more.

### YOUR ANSWER HERE — write both sentences for a coefficient of your own:

Take any two variables in `lapop_brazil.csv` (for example `trust_military` and
`govt_pride`, or `support_political_system` and `self_efficacy_political`). In the
cell below, write the **permitted descriptive sentence** for their association, then
write the **forbidden causal sentence** and cross it out with `~~strikethrough~~` —
naming the specific reason the causal verb is not earned here.

**Permitted (descriptive) sentence:**

**Forbidden (causal) sentence — struck through, with the reason:**

---

### ⚖️ Make a Design Choice

For your own project's key relationship, you will write one sentence reporting the
result. Two versions are on the table. In the cell below, choose the one your
design actually earns, and defend it in a short paragraph — say what your design
does and does not identify, and which verbs you are therefore allowed to use.

- **Option A (descriptive):** "Adjusting for [my covariates], [my regressor] and
  [my outcome] are associated by [estimate ± interval]."
- **Option B (causal):** "[My regressor] changes [my outcome] by [estimate ±
  interval]," backed by an explicit identification argument (randomization, or a
  named natural experiment).

> 💡 **Gemini Prompt:** "Here is my planned analysis: I
> will regress [outcome] on [key regressor] adjusting for [covariates], using
> observational data with no random assignment. Draft one results sentence that
> stays strictly within a descriptive/associational claim, and list the specific
> confounders that would make a causal reading of my coefficient unjustified."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check that the AI's sentence contains **no** causal verbs ("causes,"
>   "raises," "leads to," "drives") unless your design truly identifies an effect —
>   rewrite any that slipped in.
> - [ ] For each confounder it lists, confirm yourself that it plausibly affects
>   both your regressor and your outcome before you cite it — you own the mechanism.
> - [ ] Log this use in your AI ledger: tool, task, verification.

### YOUR ANSWER HERE:

**The option my design earns (A or B):**

**What my design does and does not identify, and the verbs I may use:**

---

### 📝 Practice

Four coefficient sentences. Mark each as **permitted (descriptive)**, **permitted
(causal, IF the design identifies it)**, or **forbidden (a causal claim the stated
design cannot support)**. Answer in the cell that follows.

- **A.** "In our randomized trial, assignment to the reminder text raised turnout by
  3.1 points (95% CI [1.0, 5.2])."
- **B.** "Controlling for income and education, owning a library card is associated
  with 0.4 more books read per month in this survey."
- **C.** "After controlling for study hours, being in a fraternity *causes* a 0.2
  drop in GPA (observational data)."
- **D.** "Adjusting for prior turnout, the get-out-the-vote canvass increased
  turnout by 2 points" — from a study where canvassers *chose* which houses to
  visit.


### YOUR ANSWER HERE:

**A:** &nbsp; **B:** &nbsp; **C:** &nbsp; **D:**

---

### 🎯 Project Transfer — sketch your M06 answer strategy

Time to make this your own. In the cell below, sketch the regression your project
would actually run, in the M06 template: name your **outcome**, your **key
regressor** (the relationship you care about), and **two honest covariates** you
would adjust for — "honest" meaning each is measured *before* your regressor and
plausibly confounds the relationship (not something on the causal path, not
something your regressor causes). Then write the one results sentence you expect to
report, worded inside your design's boundary.

### YOUR ANSWER HERE:

**Outcome (what varies):**

**Key regressor (the relationship I care about):**

**Two honest covariates (each pre-regressor and plausibly confounding):**

**My expected results sentence (inside my design's boundary):**

---

### 🎟️ Claim Ticket

**Claim Ticket #19** — write your project's headline **coefficient sentence** as you
would actually report it, worded strictly inside the boundary of your compass
position (descriptive unless your design truly identifies an effect). Underline the
single verb you chose and be ready to defend why your design earns it.

### YOUR ANSWER HERE:

**My coefficient sentence (inside the boundary), verb defended:**

---

## 5. The Answer Strategy Is the *Whole* Procedure

**Guiding question:** *if you quietly tried three estimators and reported the best-looking one, what was your answer strategy actually — and why is the uncertainty you printed too small?*

You have now watched an answer strategy be three different recipes for one
question: a difference in means, a regression, and a covariate-adjusted regression.
That flexibility hides the sneakiest failure of all, because every individual step
looks responsible. Suppose you run your study and try the difference-in-means; the
result is not quite significant. So you trim two outliers — reasonable — and re-run.
Still not quite. So you log-transform the skewed outcome — also reasonable — and
*now* it clears the bar. You report that last one, with its tidy standard error and
its p-value.

Here is the problem (RDSS §9.1.3, the 'Procedure' subsection): your real answer
strategy was **not** the log-transformed test. It was the entire if-then tree —
*"try spec 1; if it fails, try spec 2; if that fails, try spec 3; report whichever
clears 0.05."* That whole procedure is the recipe, and the uncertainty you printed
priced only the *last* step. It never paid for the shopping trip. Give a procedure
enough **bites at the apple** and one of them will look significant by chance alone
— so a p-value from the winning bite is far too small, and the false-alarm rate is
far higher than the 5% it advertises. The next cell makes that inflation visible in
a world where you *know* there is nothing to find.

> 💡 **Gemini Prompt:** "This cell simulates 2,000 studies of a world where
> two groups are drawn from the SAME distribution — there is no real difference. In
> each study it tests the gap three ways (raw difference, difference after trimming
> outliers, difference on the log scale) and compares the false-positive rate of one
> honest test against the rate of reporting the smallest of the three p-values:
> [paste the next cell]. Explain why testing the same data three ways and keeping
> the best inflates the false-positive rate above 5%."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Confirm from the printout that spec 1 alone lands near 5% while "report the
>   smallest of three" lands roughly twice as high — the numbers, not just the story.
> - [ ] Make sure Gemini ties the inflation to *multiple bites at the apple* (testing
>   until something clears the bar), not to any single test being wrong.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# A no-effect world: simulate many studies where the two groups are IDENTICAL,
# then see how often each recipe 'finds' a difference that isn't there (p < 0.05).
from math import erfc, sqrt

sim_rng = np.random.default_rng(SEED)   # SEED = 464; a self-contained stream for this demo
REPS, n = 2000, 60                       # 2,000 studies, n per group

def z_pvalue(a, b):
    """Two-sided p-value for a difference in means (large-sample normal approx)."""
    diff = a.mean() - b.mean()
    se = sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b))
    return erfc(abs(diff / se) / sqrt(2))           # normal two-sided tail, no scipy

def trim(x, frac=0.05):
    """Drop the lowest and highest 5% — a common 'clean the outliers' move."""
    lo, hi = np.quantile(x, [frac, 1 - frac])
    return x[(x >= lo) & (x <= hi)]

p_raw  = np.empty(REPS)   # spec 1: raw difference-in-means
p_trim = np.empty(REPS)   # spec 2: difference after trimming outliers
p_log  = np.empty(REPS)   # spec 3: difference on the log scale
for i in range(REPS):
    a = sim_rng.lognormal(3.0, 0.5, n)   # both groups from the SAME distribution:
    b = sim_rng.lognormal(3.0, 0.5, n)   # no true difference exists
    p_raw[i]  = z_pvalue(a, b)
    p_trim[i] = z_pvalue(trim(a), trim(b))
    p_log[i]  = z_pvalue(np.log(a), np.log(b))

fp_spec1 = (p_raw < 0.05).mean()                                     # one honest test
fp_best  = (np.minimum.reduce([p_raw, p_trim, p_log]) < 0.05).mean() # keep the smallest p

print("A world with NO real difference — how often each recipe cries 'significant':")
print(f"  spec 1 alone (declare it first, report it):        {fp_spec1:.3f}")
print(f"  report the smallest of the three specs (shopping): {fp_best:.3f}")
print(f"  the shopping trip multiplied the false alarms by ~{fp_best / fp_spec1:.1f}x")

assert 0.03 <= fp_spec1 <= 0.07, "one honest test should sit near the advertised 5%"
assert fp_best > 1.5 * fp_spec1, "shopping the smallest p should visibly inflate false positives"
print("✓ Self-check passed: one declared test stays near 5%; three bites at the apple roughly doubles it.")

So the false-alarm rate roughly *doubled* — not because any one test was
wrong, but because the real answer strategy was *"take three bites and keep the
best,"* and only an honest accounting of that whole procedure prices the risk
correctly. This is the same lesson the estimand/estimator/estimate triad started
with: the estimator is the **entire repeatable procedure**, exploration included,
not just the formula that produced the final number.

The fix is **not** to never explore — exploration is how good questions and
sensible specifications get found. The fix is to **declare the whole procedure**
in advance (all the if-thens, so the uncertainty pays for every bite), or to hold
out **fresh data** and confirm the winning spec there, where it gets exactly one
honest bite. That is precisely why this course made you write your answer strategy
down *before* the pilot — and it is the same **explore → declare → confirm** loop
you anchored in nb06: explore all you like, but a finding earns claim status only
once it survives a declared test it did not get to peek at first.

---

## 6. Wrap-Up

Across two meetings you assembled MIDA's final letter. You met the **triad** —
estimand, estimator, estimate; the genie's number, the recipe, the dish — and you
did the one thing real research never can: planted a true effect of 2.0, ran the
difference-in-means recipe, and measured its 0.04 miss against a truth you could
actually check. You rebuilt that recipe as a regression (the same number, exactly),
watched a balanced covariate buy precision *because* assignment was random, ran the
identical machinery on real survey data — where you read the 0.24 coefficient
strictly descriptively and struck out the causal sentence it could not support —
and finally met the subtlest failure of all: an answer strategy is the *whole*
procedure, so quietly shopping three specifications for the best-looking one
inflates your false-alarm rate, which is why you declare the procedure before you
run it.

> **"'We controlled for everything we could think of' is a confession, not a
> defense. Adjustment removes the influence of the variables you named; identity of
> a causal effect requires an argument about the ones you did not."**

Friday (nb10) opens the **generalization deep dive**: where does that standard error
actually come from, what does a confidence interval really promise, and how do you
carry a descriptive answer from your sample out to the population your frame reaches
— *with* its uncertainty instead of deleting it? You will run the same design a
thousand times and watch the histogram of estimates become your uncertainty. Bring
your M06 sketch — the answer strategy you drafted today is the spine of the full
design declaration your project is building toward.

---

## 7. Sources & Provenance

**Provenance of this notebook:**
- *RDSS ch. 9 'Choosing an answer strategy' + declaration_9.1 | the estimand/estimator/estimate triad; difference-in-means and regression as answer strategies | estimator declaration and lm_robust workflow translated R→Python (hand-rolled difference-in-means; statsmodels OLS with HC2) | translated (R→Python)*
- *RDSS ch. 9 §9.1.3 'Procedure' | the answer strategy is the whole if-then procedure; estimator-shopping ('multiple bites at the apple') inflates naive uncertainty | seeded no-effect simulation of three specifications with a shop-the-smallest-p comparison | newly-constructed-from-source-concept*
- *estimatr::lm_robust → statsmodels OLS with cov_type="HC2" | RDSS ch. 9 | the course keeps estimatr's HC2 default so numbers line up with the book | translated (R→Python)*
- *nb04 make_world (mentoring-program world) | course canonical simulation | reused verbatim; the true ATE is 2.0 by construction | reused (cross-notebook)*
- *lapop_brazil.csv | rdss package data | one adjusted descriptive association (trust in police on perceived responsiveness, adjusting for ideology) | adapted (teaching resample; the 10k-resample caveat is taught in nb06)*

**Dataset attribution:** Dataset from the `rdss` package (Blair, Coppock &
Humphreys, MIT License), companion to *Research Design in the Social Sciences*
(2023).

**Readings:**
- Blair, G., Coppock, A., & Humphreys, M. (2023). *Research Design in the Social
  Sciences*, ch. 9 'Choosing an answer strategy' (incl. §9.1.3, the 'Procedure'
  subsection). Free online: [book.declaredesign.org](https://book.declaredesign.org/).
- `statsmodels` documentation — ordinary least squares and robust (HC2) covariance
  (consult the current statsmodels docs for the OLS and `cov_type` reference).

---

<center>

Thank you!

</center>